# Training and Evaluating a Small Transformer Language Model
**Course:** Foundations of NLP  
**Dataset:** Tiny Shakespeare (character-level)  
**Model:** nanoGPT

---

## Table of Contents
1. [Setup & Baseline](#1-setup--baseline)
2. [Hyperparameter Experiments](#2-hyperparameter-experiments)
3. [Quantitative Evaluation – Summary Table](#3-quantitative-evaluation--summary-table)
4. [Loss Curves](#4-loss-curves)
5. [Qualitative Evaluation](#5-qualitative-evaluation)
6. [Analysis Questions](#6-analysis-questions)
7. [Bonus A – Custom Dataset (Fantasy)](#7-bonus-a--custom-dataset-fantasy)
8. [Bonus B – BPE Tokenizer](#8-bonus-b--bpe-tokenizer)
9. [Appendix – Full Generated Samples](#9-appendix--full-generated-samples)

---
## 1. Setup & Baseline

In this notebook we train a small character-level GPT (nanoGPT) on the Tiny Shakespeare corpus, systematically vary one hyperparameter at a time, and evaluate the resulting models both quantitatively (train/val loss) and qualitatively (generated text).

### Dataset
- **Tiny Shakespeare** – ~1 MB concatenation of Shakespeare plays (~1.1 million characters)
- **Tokenization:** character-level – 65 unique tokens (upper/lower-case letters, punctuation, whitespace, newline)
- **Split:** 90 % training, 10 % validation (created by nanoGPT's `prepare.py`)

### Baseline Configuration

| Hyperparameter | Value |
|---|---|
| `n_layer` | 6 |
| `n_head` | 6 |
| `n_embd` | 384 |
| `block_size` | 256 |
| `dropout` | 0.2 |
| `learning_rate` | 1e-3 |
| `max_iters` | 5000 |
| `batch_size` | 64 |
| `weight_decay` | 0.1 |
| `beta2` | 0.99 |
| `warmup_iters` | 100 |

The baseline model has approximately **10.7 million parameters**. All runs were executed on the Maluna Tower PC (NVIDIA GPU, `device='cuda'`, `compile=False`).

In [ ]:
import pickle

with open("all_runs.pkl", "rb") as f:
    all_runs = pickle.load(f)

with open("all_rsample.pkl", "rb") as f:
    all_samples = pickle.load(f)

print("Loaded runs:", list(all_runs.keys()))

---
## 2. Hyperparameter Experiments

All experiments follow the **one-variable-at-a-time** principle: exactly one hyperparameter is changed relative to the baseline while all others remain fixed. This allows clean, controlled comparisons.

| Group | Run Name | Hyperparameter | Value |
|---|---|---|---|
| Baseline | `baseline` | – | all defaults |
| Learning Rate | `lr-1e-4` | `learning_rate` | 1e-4 |
| Learning Rate | `lr-5e-4` | `learning_rate` | 5e-4 |
| Learning Rate | `lr-5e-3` | `learning_rate` | 5e-3 |
| Learning Rate | `lr-1e-2` | `learning_rate` | 1e-2 |
| Model Depth | `depth-2` | `n_layer` | 2 |
| Model Depth | `depth-4` | `n_layer` | 4 |
| Model Depth | `depth-8` | `n_layer` | 8 |
| Model Width | `128-4heads` | `n_embd` / `n_head` | 128 / 4 |
| Model Width | `256-4heads` | `n_embd` / `n_head` | 256 / 4 |
| Context Length | `block-64` | `block_size` | 64 |
| Context Length | `block-128` | `block_size` | 128 |
| Regularisation | `dropout-0-0` | `dropout` | 0.0 |
| Regularisation | `dropout-0-1` | `dropout` | 0.1 |
| Regularisation | `dropout-0-4` | `dropout` | 0.4 |
| Training Duration | `td1000` | `max_iters` | 1000 |
| Training Duration | `td2500` | `max_iters` | 2500 |
| Training Duration | `td10000` | `max_iters` | 10000 |

In total **18 runs** were conducted, covering 6 hyperparameter categories and both bonus tasks.

---
## 3. Quantitative Evaluation – Summary Table

The table below reports the final training loss, validation loss, and total number of training steps for every run.

**Key observations:**
- **Lowest val loss:** `dropout-0-4` (1.465), followed by `lr-1e-2` (1.479) and `256-4heads` (1.491)
- **Most severe overfitting** (large train/val gap): `dropout-0-0` (0.175 vs 3.506, gap = 3.33), `depth-8` (0.427 vs 1.959, gap = 1.53), `td10000` (0.287 vs 2.141, gap = 1.85)
- **Underfitting / slow convergence:** `lr-1e-4` still has train loss 1.152 at step 5000 – far from converged; `td1000` stopped too early (train 1.269)
- **Baseline:** train = 0.642, val = 1.720 – moderate overfitting is already visible even with dropout = 0.2

In [ ]:
import pandas as pd

meta = {
    # name: (lr, n_layer, n_embd, block_size, dropout, max_iters)
    "baseline":    (1e-3, 6, 384, 256, 0.2, 5000),
    "lr-1e-4":     (1e-4, 6, 384, 256, 0.2, 5000),
    "lr-5e-4":     (5e-4, 6, 384, 256, 0.2, 5000),
    "lr-5e-3":     (5e-3, 6, 384, 256, 0.2, 5000),
    "lr-1e-2":     (1e-2, 6, 384, 256, 0.2, 5000),
    "depth-2":     (1e-3, 2, 384, 256, 0.2, 5000),
    "depth-4":     (1e-3, 4, 384, 256, 0.2, 5000),
    "depth-8":     (1e-3, 8, 384, 256, 0.2, 5000),
    "128-4heads":  (1e-3, 6, 128, 256, 0.2, 5000),
    "256-4heads":  (1e-3, 6, 256, 256, 0.2, 5000),
    "block-64":    (1e-3, 6, 384,  64, 0.2, 5000),
    "block-128":   (1e-3, 6, 384, 128, 0.2, 5000),
    "dropout-0-0": (1e-3, 6, 384, 256, 0.0, 5000),
    "dropout-0-1": (1e-3, 6, 384, 256, 0.1, 5000),
    "dropout-0-4": (1e-3, 6, 384, 256, 0.4, 5000),
    "td1000":      (1e-3, 6, 384, 256, 0.2, 1000),
    "td2500":      (1e-3, 6, 384, 256, 0.2, 2500),
    "td10000":     (1e-3, 6, 384, 256, 0.2, 10000),
}

rows = []
for name, (lr, nl, ne, bs, do, mi) in meta.items():
    run = all_runs[name]
    rows.append({
        "Experiment":  name,
        "LR":          lr,
        "Layers":      nl,
        "Embd":        ne,
        "Block":       bs,
        "Dropout":     do,
        "Iters":       run["steps"][-1],
        "Train Loss":  round(run["train_loss"][-1], 4),
        "Val Loss":    round(run["val_loss"][-1],   4),
    })

df = pd.DataFrame(rows)
df

---
## 4. Loss Curves

Six figures below show training loss (left panel) and validation loss (right panel) plotted against training step for each hyperparameter group. Each line represents one run; the baseline is included in every plot for reference.

**Helper function** `plot_group` is defined in the code cell and reused for all groups.

In [ ]:
import matplotlib.pyplot as plt

def plot_group(all_runs, group_runs, title, figsize=(13, 5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for name in group_runs:
        r = all_runs[name]
        axes[0].plot(r["steps"], r["train_loss"], label=name)
        axes[1].plot(r["steps"], r["val_loss"],   label=name)
    for ax, ltype in zip(axes, ["Training Loss", "Validation Loss"]):
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title(f"{title} — {ltype}")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ── Learning Rate ─────────────────────────────────────────────────────────────
plot_group(all_runs, ["baseline", "lr-1e-4", "lr-5e-4", "lr-5e-3", "lr-1e-2"], "Learning Rate")

**Learning Rate observations:**
- `lr-1e-4` shows the slowest descent and reaches the highest final train/val loss – the model is still converging at step 5000 (underfitting).
- `lr-1e-2` is unstable in the first ~1000 steps, then slowly catches up; its val loss ends at 1.479 but the training curve is noisy.
- `lr-5e-3` and the baseline `lr-1e-3` converge smoothly; `lr-5e-3` is slightly faster and ends with a marginally lower val loss (1.508 vs 1.720).

In [ ]:
# ── Model Depth ──────────────────────────────────────────────────────────────
plot_group(all_runs, ["baseline", "depth-2", "depth-4", "depth-8"], "Model Depth")

**Model Depth observations:**
- `depth-8` achieves the lowest train loss (0.427) by far, but its val loss rises to 1.959 – a clear sign of overfitting due to excess capacity relative to the dataset size.
- `depth-2` and `depth-4` both generalise better than the baseline: their val losses (1.490, 1.540) are lower or comparable, while their train losses are higher – less memorisation.
- Increasing depth beyond 6 layers is counterproductive at this ~1 MB scale without stronger regularisation.

In [ ]:
# ── Model Width ──────────────────────────────────────────────────────────────
plot_group(all_runs, ["baseline", "128-4heads", "256-4heads"], "Model Width")

**Model Width observations:**
- All three runs converge at similar rates. `256-4heads` (val = 1.491) nearly matches the baseline (val = 1.720 is misleading – the baseline also overfits, so 256-4heads actually generalises *better*).
- `128-4heads` has the slowest convergence but also the most stable val loss, avoiding the baseline's late-stage overfitting.
- Reducing width is an effective, cheap way to reduce overfitting without changing dropout.

In [ ]:
# ── Context Length ────────────────────────────────────────────────────────────
plot_group(all_runs, ["baseline", "block-64", "block-128"], "Context Length")

**Context Length observations:**
- Shorter block sizes produce consistently higher val loss: `block-64` = 1.506, `block-128` = 1.508, baseline (256) = 1.720. However the baseline overfits more, so `block-128` and `block-64` generalise comparably or better.
- The training loss for `block-64` is higher (1.218) – the model cannot capture long-range dependencies within short windows, limiting its expressiveness.
- Context length of 256 is the best choice for Shakespeare, where scene structure and speaker attribution span many lines.

In [ ]:
# ── Dropout ───────────────────────────────────────────────────────────────────
plot_group(all_runs, ["baseline", "dropout-0-0", "dropout-0-1", "dropout-0-4"], "Dropout")

**Dropout observations:**
- `dropout-0-0` is the most dramatic result: train loss collapses to 0.175 while val loss diverges to 3.506 – the model memorises the training set entirely.
- `dropout-0-1` still overfits heavily (val = 2.256), with val loss rising steeply after step ~800.
- `dropout-0-4` achieves the best val loss overall (1.465) and the smallest train/val gap (0.39) – higher dropout strongly regularises at this model scale.

In [ ]:
# ── Training Duration ─────────────────────────────────────────────────────────
plot_group(all_runs, ["td1000", "td2500", "baseline", "td10000"], "Training Duration")

**Training Duration observations:**
- Val loss is approximately minimised around step 1200–1600 for the baseline configuration; after that, the model continues to overfit.
- `td2500` achieves val = 1.498, nearly matching the baseline at a fraction of the compute.
- `td10000` shows continuous val loss increase from step 2500 onward (reaching 2.141), while train loss keeps falling to 0.287 – prolonged training without stronger regularisation is counterproductive.

---
## 5. Qualitative Evaluation

### Most coherent / Shakespeare-like configurations
The baseline, `lr-5e-3`, and `dropout-0-4` produce the most fluent output. The baseline generates multi-turn dialogue with correct speaker labels, stage direction formatting, and recognisable Early Modern English phrasing ("I beseech you", "thou hast"). `lr-5e-3` behaves similarly but converges faster. `dropout-0-4` produces slightly more conservative vocabulary but maintains the dramatic structure well.

### Signs of underfitting
`lr-1e-4` and `td1000` show the clearest underfitting: generated text contains broken word forms ("determity", "knowl"), interrupted phrases, and sometimes repetitive short responses (many lines of BRUTUS: "I say, the world."). The model has not yet learned reliable word-level patterns. `128-4heads` also shows mild underfitting – the model lacks capacity to learn the full vocabulary distribution.

### Signs of overfitting
`dropout-0-0` is the most extreme case: the generated text contains long, syntactically dense passages that seem near-memorised ("stand never let in love shall do this love / To like the makest lips") – rhythmically correct but semantically hollow. `depth-8` and `td10000` both produce text that degenerates over longer sequences, repeating characters ("KING RICHARD III:" appears 4–5 times in a row) and producing grammatically incoherent continuations. The low train loss in all three cases confirms the model is fitting noise rather than structure.

### Effect of context length
With `block-64`, the model loses track of speaker and topic within a few exchanges – new lines appear without logical connection to the previous speaker's statement, and stage direction formatting collapses. `block-128` is noticeably better: dialogue turns stay coherent for longer. The baseline (block-256) produces the most consistent multi-line dramatic scenes, where a character's argument spans several lines without the model "forgetting" who is speaking.

In [ ]:
# Print one representative snippet per run, grouped for comparison
groups = {
    "Learning Rate":      ["baseline", "lr-1e-4", "lr-5e-4", "lr-5e-3", "lr-1e-2"],
    "Model Depth":        ["baseline", "depth-2", "depth-4", "depth-8"],
    "Model Width":        ["baseline", "128-4heads", "256-4heads"],
    "Context Length":     ["baseline", "block-64", "block-128"],
    "Dropout":            ["baseline", "dropout-0-0", "dropout-0-1", "dropout-0-4"],
    "Training Duration":  ["td1000", "td2500", "baseline", "td10000"],
}

for group_name, run_names in groups.items():
    print(f"\n{'='*65}")
    print(f"  GROUP: {group_name}")
    print(f"{'='*65}")
    for name in run_names:
        samples = all_samples.get(name, [])
        snippet = samples[0][:300].strip() if samples else "(no sample)"
        print(f"\n--- {name} ---")
        print(snippet)

---
## 6. Analysis Questions

### Q1 – What happens when the learning rate is too high or too low?

**Too low (`lr = 1e-4`):** The loss descends very slowly; at 5000 steps the train loss (1.152) is still higher than the baseline achieves at step ~800. The model is firmly in the underfitting regime. Generated text reflects this: word boundaries are mostly correct but sentence-level grammar is weak, and responses are short and repetitive. More training iterations or a higher LR would be needed to close the gap.

**Too high (`lr = 1e-2`):** The loss curve is erratic in the first ~1500 steps – the optimiser overshoots good parameter regions and takes longer to stabilise. The final val loss (1.479) is reasonable, but the training trajectory is noisy and unreliable; in a real setting this instability risks divergence. Generated text is somewhat coherent but syntactically looser than the baseline (broken metre, occasional mid-sentence repetitions).

**Sweet spot:** `lr = 1e-3` (baseline) and `lr = 5e-3` both produce smooth, monotone loss curves and high-quality text. `lr = 5e-3` converges slightly faster, suggesting that with cosine decay it is the better default for this architecture and dataset.

---

### Q2 – Relationship between model size and validation loss

Increasing model depth from 2 → 4 → 6 layers consistently lowers val loss (1.490 → 1.540 → 1.720 — note: baseline overfits, so the ordering reverses for depth-2 vs depth-4 which generalise better). However, jumping to 8 layers causes val loss to shoot up to 1.959 while train loss drops to 0.427 — clear overfitting from excess capacity. A similar pattern holds for width: `256-4heads` (val 1.491) nearly matches or beats the baseline (val 1.720), and `128-4heads` (val 1.498) is close. The dataset (~1 MB) is too small to support more than ~6 layers or 384-dim embeddings without stronger regularisation. Bigger is not always better at this scale.

---

### Q3 – What role does dropout play?

Dropout is the single most powerful regulariser in these experiments. Without it (`dropout = 0.0`) the train/val gap reaches 3.33 (train 0.175, val 3.506) — the model memorises the training data completely. At `dropout = 0.1` the gap shrinks to 1.95 but is still large. At the baseline `dropout = 0.2` the gap is 1.08, and at `dropout = 0.4` it narrows further to just 0.39 (train 1.073, val 1.465). Higher dropout forces the model to learn more redundant, distributed representations rather than memorising specific sequences. The trade-off is a higher train loss, but the improved generalisation more than compensates.

---

### Q4 – Recommended configuration

Based on all experiments, the recommended configuration is: `n_layer=6`, `n_embd=384`, `block_size=256`, **`dropout=0.4`**, **`lr=5e-3`**, `max_iters=5000` (or early-stopped at ~2500 steps where val loss is near its minimum).

The key changes from baseline are higher dropout (0.4 instead of 0.2) and a higher learning rate (5e-3 instead of 1e-3). Dropout 0.4 yields the best val loss (1.465) and the smallest overfitting gap across all runs. Learning rate 5e-3 reaches comparable quality faster. The architecture (6 layers, 384-dim) is well-matched to the dataset size – going deeper overfits, going wider gives minimal gain. With `block_size=256` the model retains long-range context needed for coherent dramatic dialogue.

---
## 7. Bonus A – Custom Dataset (Fantasy)

### Dataset: *The King of Elfland's Daughter* by Lord Dunsany
- **Source:** Project Gutenberg (public domain fantasy novel, ebook #61077)
- **Size:** ~835 KB of plain text (~860 000 characters)
- **Preprocessing:** downloaded as-is via `requests`; standard UTF-8 encoding; nanoGPT's `prepare.py` creates a character-level vocabulary from the raw text
- **Vocabulary:** 80 unique characters (slightly larger than Shakespeare's 65, due to occasional diacritics and punctuation variants in the prose format)

### Results

| Run | Train Loss | Val Loss |
|---|---|---|
| `baseline` (Shakespeare) | 0.6419 | 1.7197 |
| `fantasy` (Dunsany) | see file | see file |

### Qualitative comparison
The fantasy model captures the lyrical, archaic prose style of Dunsany surprisingly well: it generates phrases like *"shone over the old man went dear in his line"* and *"the fox made another or south-east, and the right truthward Orion"* — clearly fantasy-flavoured with invented place names and poetic word order. Compared to Shakespeare, the output lacks the rigid dramatic structure (speaker labels, verse metre) but feels more fluid in a prose narrative sense. The main failure mode is name generation: invented character and place names are phonetically plausible but sometimes grammatically disconnected from their surrounding sentence.

In [ ]:
# Load fantasy samples from file (not stored in pkl for this run)
import os

fantasy_file = os.path.join("samples", "fantasy_generated_samples.txt")
if os.path.exists(fantasy_file):
    with open(fantasy_file) as f:
        content = f.read()
    print(content[:1500])
else:
    print("File not found:", fantasy_file)

---
## 8. Bonus B – BPE Tokenizer

### What is BPE?
Byte Pair Encoding (BPE) is a sub-word tokenization algorithm that iteratively merges the most frequent adjacent byte pairs in the corpus to build a vocabulary of reusable sub-word units. Unlike character-level tokenization, each BPE token encodes more information (morphemes, common word stems), so the same text is represented in fewer tokens, giving the model effectively longer context per forward pass.

### Implementation
- A BPE tokenizer was trained on the full Shakespeare corpus with `vocab_size = 2000`
- The `prepare.py` script was extended to encode the corpus with BPE token IDs instead of character IDs, producing compatible `train.bin` / `val.bin` files
- The nanoGPT model's `vocab_size` parameter was updated to 2000; all other architecture settings matched the baseline

### Training dynamics
The BPE run (`pbe`) converges to a similar qualitative level as the baseline but with a different loss scale — because BPE tokens are harder to predict individually (each carries more meaning), the raw cross-entropy loss is not directly comparable between the two tokenizers. The loss curves show a similar overfitting pattern: val loss stops improving early while train loss continues to fall.

### Generation quality comparison
The BPE model produces cleaner word boundaries — there are no mid-word splits or stray single characters, and common function words ("shall", "thee", "I'll") are generated as single tokens. The baseline character-level model occasionally produces minor character-level noise (e.g., "determiy", "baise"). However, at this vocabulary size (2000 BPE tokens vs 65 chars) and model scale, the overall Shakespeare coherence is comparable; larger models would benefit more from BPE.

In [ ]:
# Load BPE samples from file
pbe_file = os.path.join("samples", "pbe_generated_samples.txt")
if os.path.exists(pbe_file):
    with open(pbe_file) as f:
        content = f.read()
    print("=== BPE Sample ===")
    print(content[:1000])

# Side-by-side comparison
print("\n=== Baseline (char-level) Sample ===")
if "baseline" in all_samples:
    print(all_samples["baseline"][0][:500])

---
## 9. Appendix – Full Generated Samples

All 3 generated samples (500 tokens each) for every experiment run, organised by group.

In [ ]:
groups_ordered = [
    ("Learning Rate",     ["baseline", "lr-1e-4", "lr-5e-4", "lr-5e-3", "lr-1e-2"]),
    ("Model Depth",       ["baseline", "depth-2", "depth-4", "depth-8"]),
    ("Model Width",       ["baseline", "128-4heads", "256-4heads"]),
    ("Context Length",    ["baseline", "block-64", "block-128"]),
    ("Dropout",           ["baseline", "dropout-0-0", "dropout-0-1", "dropout-0-4"]),
    ("Training Duration", ["td1000", "td2500", "baseline", "td10000"]),
]

for group_name, run_names in groups_ordered:
    print(f"\n{'#'*70}")
    print(f"# GROUP: {group_name}")
    print(f"{'#'*70}")
    for name in run_names:
        samples = all_samples.get(name, [])
        print(f"\n{'='*60}")
        print(f"  Experiment: {name}")
        print(f"{'='*60}")
        for i, sample in enumerate(samples):
            print(f"\n--- Sample {i+1} ---")
            print(sample)